# Word Embedding Model

> **Words that appear in similar contexts should have similar vectors.**

## 1. Dataset

In [ ]:
import numpy as np

corpus = [
    "he is a boy",     "she is a girl",
    "he is like him",  "she is like her",
    "boy is like him", "girl is like her",
]

| Word | Appears with           |
| ---- | ---------------------- |
| he   | a, boy, him, is, like  |
| she  | a, girl, her, is, like |
| boy  | a, he, him, is, like   |
| girl | a, her, is, like, she  |

**he and boy share context -> similar meaning. she and girl share context -> similar meaning.**

## 2. Vocabulary

Computers work with numbers not words.

In [ ]:
vocab = sorted(set(word for s in corpus for word in s.split()))
w2i = {w: i for i, w in enumerate(vocab)}

print("Vocabulary:", vocab)
print("Word to index:", w2i)

## 3. Context profiles

For every word, store all other words appearing in the same sentence. This is called a **distributional context profile**.

> Words used in similar contexts have similar meanings. (Distributional hypothesis)

In [ ]:
context_of = {w: set() for w in vocab}
for sentence in corpus:
    words = sentence.split()
    for w1 in words:
        for w2 in words:
            if w1 != w2:
                context_of[w1].add(w2)

print("Context profiles:")
for word in vocab:
    print(f"  {word:7s} -> {sorted(context_of[word])}")

## 4. Context similarity

**Jaccard similarity** = shared / all_context

We use `.intersection()` to find shared context words and `.union()` to find all context words.

Example for `he` and `boy`:
```
he  context: {a, boy, him, is, like}
boy context: {a, he, him, is, like}

shared      = intersection -> {a, him, is, like}          -> 4 words
all_context = union        -> {a, boy, he, him, is, like} -> 6 words

similarity = len(shared) / len(all_context) = 4 / 6 = 0.67
```

In [ ]:
pairs = []
for i in range(len(vocab)):
    for j in range(i + 1, len(vocab)):
        word_a = vocab[i]
        word_b = vocab[j]
        shared = context_of[word_a].intersection(context_of[word_b])
        all_context = context_of[word_a].union(context_of[word_b])
        similarity = len(shared) / len(all_context)
        pairs.append((i, j, similarity))

print("Context similarities:")
for i, j, sim in pairs:
    print(f"  {vocab[i]:7s} - {vocab[j]:7s} = {sim:.2f}")

## 5. Embedding class

Each word gets a **2D vector**. Training rule:
- **similarity > 0.5 -> pull closer**
- **similarity <= 0.5 -> push apart**

In [ ]:
class Embedding:
    def __init__(self, n_words, learning_rate=0.05, n_epochs=1000):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.embeddings = np.random.randn(n_words, 2) * 0.5

    def train(self, pairs):
        for epoch in range(self.n_epochs):
            for i, j, sim in pairs:
                diff = self.embeddings[i] - self.embeddings[j]
                distance = np.sqrt(np.sum(diff ** 2)) + 1e-8
                direction = diff / distance
                if sim > 0.5:  # similar contexts -> pull closer
                    self.embeddings[i] -= self.learning_rate * sim * direction
                    self.embeddings[j] += self.learning_rate * sim * direction
                else:  # different contexts -> push apart
                    self.embeddings[i] += self.learning_rate * (1 - sim) * direction
                    self.embeddings[j] -= self.learning_rate * (1 - sim) * direction

    def get_embedding(self, index):
        return self.embeddings[index]

## 6. Distance function

**Euclidean distance**: lower = more similar, higher = different.

In [ ]:
def distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

## 7. Train and results

In [ ]:
np.random.seed(123)
model = Embedding(n_words=len(vocab))
model.train(pairs)

print("Embeddings:")
for word in ["he", "she", "boy", "girl"]:
    v = model.get_embedding(w2i[word])
    print(f"  {word:7s} -> [{v[0]:+.4f}, {v[1]:+.4f}]")

print("\nDistance:")
print("he  - boy  =", f"{distance(model.get_embedding(w2i['he']), model.get_embedding(w2i['boy'])):.2f}")
print("she - girl =", f"{distance(model.get_embedding(w2i['she']), model.get_embedding(w2i['girl'])):.2f}")
print("he  - she  =", f"{distance(model.get_embedding(w2i['he']), model.get_embedding(w2i['she'])):.2f}")
print("boy - girl =", f"{distance(model.get_embedding(w2i['boy']), model.get_embedding(w2i['girl'])):.2f}")
print("he  - girl =", f"{distance(model.get_embedding(w2i['he']), model.get_embedding(w2i['girl'])):.2f}")
print("she - boy  =", f"{distance(model.get_embedding(w2i['she']), model.get_embedding(w2i['boy'])):.2f}")

The model learned: `he ~ boy` (close), `she ~ girl` (close), and all cross-pairs are far apart.

This is a **mini Word2Vec without neural networks**. Same principle behind `king - man + woman ~ queen`.